In [ ]:
pip install -U "transformers>=5.10.1" accelerate safetensors sentencepiece peft pandas unsloth

In [ ]:
import json
import re
import torch
import bitsandbytes as bnb
import pandas as pd

from collections import Counter
from tqdm.auto import tqdm
from unsloth import FastModel

BASE_MODEL = "unsloth/gemma-4-E2B-it"
ADAPTER_PATH = "models/gemma-4-e2b-it-best"
MAX_SEQ_LENGTH = 2048

# Select one: base_bf16, base_4bit, qlora_bf16, or qlora_4bit.
CONFIG = "base_4bit"

EXPERIMENTS = {
    "base_bf16": {
        "model_name": BASE_MODEL,
        "load_in_4bit": False,
        "label": "Gemma 4 E2B Base (BF16)",
    },
    "base_4bit": {
        "model_name": BASE_MODEL,
        "load_in_4bit": True,
        "label": "Gemma 4 E2B Base (4-bit)",
    },
    "qlora_bf16": {
        "model_name": ADAPTER_PATH,
        "load_in_4bit": False,
        "label": "Gemma 4 E2B + QLoRA (BF16)",
    },
    "qlora_4bit": {
        "model_name": ADAPTER_PATH,
        "load_in_4bit": True,
        "label": "Gemma 4 E2B + QLoRA (4-bit)",
    },
}

if CONFIG not in EXPERIMENTS:
    raise ValueError(f"Unknown configuration: {CONFIG}")

experiment = EXPERIMENTS[CONFIG]
model, tokenizer = FastModel.from_pretrained(
    model_name=experiment["model_name"],
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=experiment["load_in_4bit"],
    dtype=torch.bfloat16,
)
model.eval()

four_bit_layers = [
    module for module in model.modules()
    if isinstance(module, bnb.nn.Linear4bit)
]
expected_four_bit = experiment["load_in_4bit"]
assert bool(four_bit_layers) == expected_four_bit, (
    "Loaded precision does not match the selected experiment."
)

print("Experiment:", experiment["label"])
print("Model path:", experiment["model_name"])
print("4-bit layers:", len(four_bit_layers))
print("Parameter dtypes:", Counter(str(p.dtype) for p in model.parameters()))

/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:86: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchaudio/lib/libtorchaudio.so
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0917 14:29:27.979000 267615 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0917 14:29:28.009000 267615 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:2189: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


==((====))==  Unsloth 2026.9.5: Fast Llama patching. Transformers: 5.10.1.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded: unsloth/gemma-4-E2B-it-best


In [ ]:
# Load the local evaluation dataset
DATASET_PATH = "datasets/usmle-self-assessment-tetun/usmle_step_2_tetun.json"
with open(DATASET_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

print("Number of questions:", len(dataset))
print(dataset[0])

Number of questions: 109
{'question': "Ema-mane ida ho tinan 68 lori mai ba departamentu emerjénsia tanba istória loron 2 ho letarjia progresiva no tremores. Nia ho alkoholizmu no dezordem uzu opiátu, kokaína no anfetamina, maibé nia abstén tiha ona husi álkol no substánsia ilegál durante semana rua liuba. Nia mós ho sirróze alkoólika. Nia aimoruk ida de'it mak multivitamina diária. Nia uluk hemu servejada 12-oz neen loron-loron. Nia parese moras króniku no somnolentu. Temperatura mak 36.8°C (98.2°F), pulsu mak 100/min, respirasaun mak 18/min, no presaun-raan mak 100/60 mm Hg. Oksimetria pulsu iha anin kuartu nian hatudu saturasaun oksijéniu 94%. Ezame hatudu ikteru konjuntivál, asite, no edema 2+ iha extremidade inferior sira. Nervu kraniál sira intactu. Forsa muskular mak 4/5 iha parte hotu. Bainhira nia matan-moris, iha tremor posturál no asaun nian husi liman-fuan no liman ne'ebé akompaña ho lakon forsa muskular ne'ebé derrepente no badak. Nia ko'alia normál, no nia bele tuir instr

In [ ]:
# Build the multiple-choice prompt
def build_prompt(item):
    options = item["options"]
    if isinstance(options, dict):
        options_text = "\n".join(
            f"{label}: {text}"
            for label, text in options.items()
            if text is not None
        )
    else:
        options_text = "\n".join(options)

    return f"""Choose the correct answer to the following multiple-choice question.

Question:
{item["question"]}

Options:
{options_text}

IMPORTANT OUTPUT RULES:
- Return exactly ONE uppercase answer label, which is the key value.
- Valid labels are: 'A', 'B', 'C', 'D', 'E', 'F', or 'G'.
- Output the label only.
- Do not include thoughts, reasoning, analysis, explanations, or intermediate steps.
- Do not output words such as "thought", "answer", or "response".
- Do not provide any explanatory text or answer content beyond the required output.
- Do not add punctuation, words, spaces, or any other text.

Your entire response must be exactly one of:
A
B
C
D
E
F
G

Answer:"""


print(build_prompt(dataset[0]))

Choose the correct answer to the following multiple-choice question.

            Question:
            Ema-mane ida ho tinan 68 lori mai ba departamentu emerjénsia tanba istória loron 2 ho letarjia progresiva no tremores. Nia ho alkoholizmu no dezordem uzu opiátu, kokaína no anfetamina, maibé nia abstén tiha ona husi álkol no substánsia ilegál durante semana rua liuba. Nia mós ho sirróze alkoólika. Nia aimoruk ida de'it mak multivitamina diária. Nia uluk hemu servejada 12-oz neen loron-loron. Nia parese moras króniku no somnolentu. Temperatura mak 36.8°C (98.2°F), pulsu mak 100/min, respirasaun mak 18/min, no presaun-raan mak 100/60 mm Hg. Oksimetria pulsu iha anin kuartu nian hatudu saturasaun oksijéniu 94%. Ezame hatudu ikteru konjuntivál, asite, no edema 2+ iha extremidade inferior sira. Nervu kraniál sira intactu. Forsa muskular mak 4/5 iha parte hotu. Bainhira nia matan-moris, iha tremor posturál no asaun nian husi liman-fuan no liman ne'ebé akompaña ho lakon forsa muskular ne'eb

In [38]:
# ============================================================
# 6. Extract answer label
# ============================================================

def extract_answer(raw_response):

    text = raw_response.strip().upper()

    # Find valid standalone answer labels
    matches = re.findall(
        r"\b([A-G])\b",
        text
    )

    # For Gemma 4 outputs such as:
    # thought
    # A
    #
    # use the final valid label
    if matches:
        return matches[-1]

    return None

In [39]:
# Check the GPU
import torch

free, total = torch.cuda.mem_get_info()

print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"Free:      {free/1024**3:.2f} GB")

Allocated: 5.72 GB
Reserved:  6.34 GB
Free:      16.69 GB


In [ ]:
# Predict one question
def predict_mcq(model, tokenizer, item):
    prompt = build_prompt(item)
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        enable_thinking=False,
        return_tensors="pt",
    )
    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            use_cache=True,
            do_sample=False,
        )

    generated_tokens = outputs[0][input_length:]
    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()
    return extract_answer(raw_response), raw_response

In [ ]:
# Test one questions
item = dataset[0]

prediction, raw_response = predict_mcq(
    model=model,
    tokenizer=tokenizer,
    item=item,
)

print("Question:")
print(item["question"])

print("\nGold answer:")
print(item["answer_idx"])

print("\nRaw model response:")
print(raw_response)

print("\nExtracted prediction:")
print(prediction)

In [42]:
# ============================================================
# 8. Evaluate complete dataset
# ============================================================

def evaluate_mcq(
    model,
    tokenizer,
    dataset,
):

    results = []

    for i, item in enumerate(
        tqdm(
            dataset,
            desc="Evaluating"
        )
    ):

        prediction, raw_response = predict_mcq(
            model=model,
            tokenizer=tokenizer,
            item=item,
        )

        # Gold label directly from dataset
        gold_label = str(
            item["answer_idx"]
        ).strip().upper()

        # Compare predicted label with gold label
        correct = prediction == gold_label

        results.append({
            "index": i,
            "question": item["question"],
            "gold_label": gold_label,
            "prediction": prediction,
            "correct": correct,
            "raw_response": raw_response,
        })

    results_df = pd.DataFrame(results)

    return results_df

In [ ]:
# ============================================================
# 9. Run evaluation
# ============================================================

results_df = evaluate_mcq(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
)

In [44]:
# Preview results
results_df.head(10)

,index,question,gold_label,prediction,correct,raw_response
0,0,Ema-mane ida ho tinan 68 lori mai ba departame...,C,A,False,A
1,1,Feto ida ho tinan 45 mai iha departamentu emer...,F,A,False,A
2,2,"Feto tinan 87, grávida 6, para 6, lori ba depa...",D,A,False,A
3,3,Feto ida ho tinan 25 admitidu iha ospitál tanb...,D,A,False,A
4,4,Labarik mane tinan 16 ida lori ba departamentu...,A,A,True,A
5,5,"Feto tinan 29 ida, ne'ebé uluk saudavel, tama ...",A,A,True,A
6,6,Feto primigrávida ida ho tinan 27 mai ba médik...,D,A,False,A
7,7,Investigadór ida hakarak hatun insidénsia diab...,E,A,False,A
8,8,Ema mane ida ho idade tinan 70 mai klínika tan...,C,A,False,A
9,9,Feto ida ho tinan 52 bá médiku tanba istória l...,E,A,False,A


In [ ]:
# ============================================================
# 10. Metrics
# ============================================================

total = len(results_df)

correct = results_df[
    "correct"
].sum()

accuracy = results_df[
    "correct"
].mean()

invalid = results_df[
    "prediction"
].isna().sum()

invalid_rate = invalid / total


print(f"Model: {experiment['label']}")

print(f"Total questions: {total}")

print(f"Correct answers: {correct}")

print(
    f"Accuracy: "
    f"{accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Invalid responses: "
    f"{invalid}"
)

print(
    f"Invalid rate: "
    f"{invalid_rate * 100:.2f}%"
)

In [ ]:
# Inspect invalid responses
results_df[
    results_df["prediction"].isna()
][
    [
        "index",
        "question",
        "gold_label",
        "raw_response",
    ]
]